### Data Loading

In [1]:
from datasets import load_dataset
dataset_raw= load_dataset("lfcc/portuguese_ner")
dataset_raw


c:\Users\Maria\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 3716
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 930
    })
})

In [2]:
dataset_raw["train"].features

{'tokens': List(Value('string')),
 'ner_tags': List(ClassLabel(names=['O', 'B-Data', 'I-Data', 'B-Local', 'I-Local', 'B-Organizacao', 'I-Organizacao', 'B-Pessoa', 'I-Pessoa', 'B-Profissao', 'I-Profissao']))}

### Data Pre-Processing

In [3]:
from transformers import AutoTokenizer
#usar o tokanaizer usado para treinar o modelo que tamos a usar ?
tokenizer=AutoTokenizer.from_pretrained("neuralmind/bert-base-portuguese-cased")

In [4]:
inputs=tokenizer("As aulas de PLN são muito interessantes!")
inputs

{'input_ids': [101, 510, 6880, 125, 212, 22327, 22320, 453, 785, 20764, 106, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [5]:
tokens=tokenizer.convert_ids_to_tokens(inputs["input_ids"])
print(tokens)
#cls trieno do modelo, classifica se as frases fazem sentido ou não
#sep serpara as frases

['[CLS]', 'As', 'aulas', 'de', 'P', '##L', '##N', 'são', 'muito', 'interessantes', '!', '[SEP]']


In [6]:
dataset_raw["train"]["tokens"]

Column([['Filiação', ':', 'Antonio', 'Joaquim', 'Aguiar', 'e', 'Engracia', 'Maria', '.', 'Natural', 'e/ou', 'residente', 'em', 'CUNHA', ',', 'Santa', 'Maria', ',', 'actual', 'concelho', 'de', 'PAREDES', 'COURA', 'e', 'distrito', '(', 'ou', 'país', ')', 'Viana', 'do', 'Castelo', '.'], ['Filiação', ':', 'Domingos', 'Pires', 'e', 'Comba', 'Fernandes', '.', 'Natural', 'e/ou', 'residente', 'em', 'VALONGO', 'MILHAIS', ',', 'Sao', 'Goncalo', ',', 'actual', 'concelho', 'de', 'MURCA', 'e', 'distrito', '(', 'ou', 'país', ')', 'VILA', 'REAL', '.'], ['Termo', 'de', 'justificação', 'do', 'baptismo', 'de', 'Pedro', 'Gonçalves', 'Coques', ',', 'nascido', 'em', '29.06.1876', 'e', 'baptizado', '"', '(', '…', ')', 'por', 'dias', 'do', 'mês', 'de', 'Julho', 'do', 'dito', 'ano', ',', '(', '…', ')', '"', ',', 'na', 'igreja', 'do', 'Jardim', 'do', 'Mar', ',', 'Calheta', '.'], ['Doc.danificado', '.'], ['1898-11-01', '/', '1898-11-01'], ...])

In [7]:
tokens=["as","aulas","plneb","são","interessantes","!"]
inputs=tokenizer(tokens, is_split_into_words=True)
newtokens=tokenizer.convert_ids_to_tokens(inputs["input_ids"])
print(newtokens)

['[CLS]', 'as', 'aulas', 'pl', '##ne', '##b', 'são', 'interessantes', '!', '[SEP]']


In [8]:
len(tokens), len(newtokens)

(6, 10)

In [9]:
inputs.word_ids() #mapeamento entre tokens antigos e tokens novos

[None, 0, 1, 2, 2, 2, 3, 4, 5, None]

In [10]:
print(dataset_raw)

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 3716
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 930
    })
})


In [11]:
def align_labels_with_tokens(word_ids,labels):
    new_labels=[]
    previous_word= None
    for word_id in word_ids:
        if word_id == None:
            new_labels.append(-100)#codigo para igonorar a label
        elif previous_word != word_id:
            new_labels.append(labels[word_id])#mantem a label
        else:
            new_labels.append(-100)#ignorar as sub words
        previous_word= word_id
    return new_labels

def tokenize_dataset(dataset):
    res=[]
    for row in dataset:
        inputs=tokenizer(row["tokens"], is_split_into_words=True)
        new_labels=align_labels_with_tokens(inputs.word_ids(),row["ner_tags"])
        inputs["labels"]= new_labels
        res.append(inputs)
    return res

train_data= tokenize_dataset(dataset_raw["train"])
test_data= tokenize_dataset(dataset_raw["test"])
len(train_data), len(test_data)

    

(3716, 930)

In [12]:
from datasets import Dataset
train_dataset=Dataset.from_list(train_data)
test_dataset=Dataset.from_list(test_data)

print(train_dataset)
print(test_dataset)

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 3716
})
Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 930
})


In [13]:
from transformers import AutoModelForTokenClassification

model= AutoModelForTokenClassification.from_pretrained("neuralmind/bert-base-portuguese-cased")

c:\Users\Maria\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Maria\.cache\huggingface\hub\models--neuralmind--bert-base-portuguese-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 19710.36it/s]
[trans